## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [85]:
import seaborn as sns
import pandas as pd
import numpy as np


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
360,0,3,male,40.0,1,4,27.9000,S,Third,man,True,NaN,Southampton,no,False
218,1,1,female,32.0,0,0,76.2917,C,First,woman,False,D,Cherbourg,yes,True
216,1,3,female,27.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
474,0,3,female,22.0,0,0,9.8375,S,Third,woman,False,NaN,Southampton,no,True
843,0,3,male,34.5,0,0,6.4375,C,Third,man,True,NaN,Cherbourg,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [92]:
missed_data = pd.Series({col: titanic_data[col].isnull().sum() for col in titanic_data.columns})
missed_data

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [87]:
titanic_data = titanic_data.dropna(axis="columns", thresh=titanic_data.shape[0] // 2 + 1)
titanic_data = titanic_data.dropna(thresh=titanic_data.shape[1] // 2 + 1)

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [89]:
fill_vals = titanic_data["who"].map(
    {
        "man": titanic_data[titanic_data.who == "man"]["age"].median(),
        "woman": titanic_data[titanic_data.who == "woman"]["age"].median(),
        "child": titanic_data[titanic_data.who == "child"]["age"].median(),
    }
)
titanic_data["age"] = titanic_data["age"].fillna(fill_vals)

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [91]:
titanic_data = titanic_data.dropna(thresh=len(titanic_data.columns) - 1)

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [93]:
towns = titanic_data["embark_town"].value_counts()
towns.index[0]

'Southampton'

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [94]:
survivos_percent = (titanic_data["survived"].mean() * 100).round(2)
survivos_percent

38.25

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [95]:
survivos_town = titanic_data["embark_town"][titanic_data["survived"] == 1].value_counts()
survivos_town

embark_town
Southampton    217
Cherbourg       93
Queenstown      30
Name: count, dtype: int64

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [96]:
survivos_class = (
    titanic_data["class"][titanic_data["survived"] == 1].value_counts()
    / titanic_data["class"].value_counts()
    * 100
).round(2)
survivos_class

class
First     62.62
Second    47.28
Third     24.24
Name: count, dtype: float64

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [97]:
wealthy_percent = (titanic_data[titanic_data["fare"] >= 100]["survived"].mean() * 100).round(2)
wealthy_percent

73.58

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [98]:
child_alone = titanic_data[(titanic_data["who"] == "child") & (titanic_data["alone"] == True)]
len(child_alone)

6

Какие выводы вы можете сделать о выживших пассажирах Титаника? 